## 03_ML

- 02_DataProcess.ipynb에서 분리해 둔 학습/검증/테스트 데이터를 이용
- 대상: `data/diabetes_imputed_*_{train,validation,test}.csv` (target: `Outcome`)
- 결과: 학습된 모델(.pkl)과 검증/테스트 예측 결과(.csv)를 저장하여 `04_Result.ipynb`에서 바로 불러와 평가/시각화 진행

## 1. 환경 설정과 데이터 로드

In [13]:
from pathlib import Path
import pickle
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

In [14]:
DATA_DIR = Path("data")
TARGET_COL = "Outcome"
DATASETS = {
    "median": {
        "train": DATA_DIR / "diabetes_imputed_median_train.csv",
        "validation": DATA_DIR / "diabetes_imputed_median_validation.csv",
        "test": DATA_DIR / "diabetes_imputed_median_test.csv",
    },
    "iterative": {
        "train": DATA_DIR / "diabetes_imputed_iterative_train.csv",
        "validation": DATA_DIR / "diabetes_imputed_iterative_validation.csv",
        "test": DATA_DIR / "diabetes_imputed_iterative_test.csv",
    },
}

model_dir = Path("models")
pred_dir = Path("predictions")
model_dir.mkdir(exist_ok=True)
pred_dir.mkdir(exist_ok=True)


def load_dataset(version: str = "median"):
    if version not in DATASETS:
        raise ValueError(f"지원하지 않는 데이터 버전: {version}")
    paths = DATASETS[version]
    data = {split: pd.read_csv(path) for split, path in paths.items()}
    for split, df in data.items():
        if TARGET_COL not in df.columns:
            raise ValueError(f"{path}에 '{TARGET_COL}' 컬럼이 없습니다.")
    return data


def split_features_labels(df):
    X = df.drop(columns=[TARGET_COL])
    y = df[TARGET_COL]
    return X, y


# 모든 버전을 한 번에 적재
datasets = {name: load_dataset(name) for name in DATASETS}
for name, parts in datasets.items():
    print(f"[{name}] train={parts['train'].shape}, validation={parts['validation'].shape}, test={parts['test'].shape}")


[median] train=(460, 9), validation=(154, 9), test=(154, 9)
[iterative] train=(460, 9), validation=(154, 9), test=(154, 9)


In [15]:
def get_model_builders():
    builders = {
        "log_reg": lambda: Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=300, n_jobs=-1, class_weight="balanced")),
        ]),
        "knn": lambda: Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=5)),
        ]),
        "random_forest": lambda: RandomForestClassifier(
            n_estimators=300, random_state=42, class_weight="balanced", n_jobs=-1
        ),
        "gradient_boosting": lambda: GradientBoostingClassifier(random_state=42),
        "xgboost" : lambda: XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42)
    }
    return builders


In [16]:
def compute_scores(model, X, y):
    y_pred = model.predict(X)
    scores = {
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
    }

    y_score = None
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X)[:, 1]
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X)

    if y_score is not None:
        try:
            scores["roc_auc"] = roc_auc_score(y, y_score)
        except ValueError:
            scores["roc_auc"] = np.nan
    else:
        scores["roc_auc"] = np.nan
    return scores, y_pred, y_score


def train_and_save(dataset_name: str, splits: dict):
    builders = get_model_builders()
    results = []

    X_train, y_train = split_features_labels(splits["train"])
    X_val, y_val = split_features_labels(splits["validation"])
    X_test, y_test = split_features_labels(splits["test"])

    for model_name, builder in builders.items():
        model = builder()
        model.fit(X_train, y_train)

        train_scores, _, _ = compute_scores(model, X_train, y_train)
        val_scores, val_pred, val_score = compute_scores(model, X_val, y_val)
        test_scores, test_pred, test_score = compute_scores(model, X_test, y_test)

        # 결과 모음
        row = {
            "dataset": dataset_name,
            "model": model_name,
            **{f"train_{k}": v for k, v in train_scores.items()},
            **{f"val_{k}": v for k, v in val_scores.items()},
            **{f"test_{k}": v for k, v in test_scores.items()},
        }
        results.append(row)

        # 모델 저장 (pickle)
        model_path = model_dir / f"{dataset_name}_{model_name}.pkl"
        with open(model_path, "wb") as f:
            pickle.dump(model, f)

        # 예측값 저장 (검증/테스트)
        for split_name, y_true, y_pred, y_score in [
            ("validation", y_val, val_pred, val_score),
            ("test", y_test, test_pred, test_score),
        ]:
            pred_df = pd.DataFrame({
                "y_true": y_true,
                "y_pred": y_pred,
            })
            if y_score is not None:
                pred_df["score"] = y_score
            out_path = pred_dir / f"{dataset_name}_{model_name}_{split_name}_pred.csv"
            pred_df.to_csv(out_path, index=False)

        print(f"[{dataset_name}] {model_name} 완료: 저장 -> {model_path.name}")

    return pd.DataFrame(results)


In [17]:
# 모든 데이터셋/모델 학습 실행
all_results = []
for ds_name, ds_splits in datasets.items():
    res_df = train_and_save(ds_name, ds_splits)
    all_results.append(res_df)

summary_df = pd.concat(all_results, ignore_index=True)
summary_df.sort_values(["dataset", "val_f1"], ascending=[True, False], inplace=True)
summary_df


[median] log_reg 완료: 저장 -> median_log_reg.pkl
[median] knn 완료: 저장 -> median_knn.pkl
[median] random_forest 완료: 저장 -> median_random_forest.pkl
[median] gradient_boosting 완료: 저장 -> median_gradient_boosting.pkl
[median] xgboost 완료: 저장 -> median_xgboost.pkl
[iterative] log_reg 완료: 저장 -> iterative_log_reg.pkl
[iterative] knn 완료: 저장 -> iterative_knn.pkl
[iterative] random_forest 완료: 저장 -> iterative_random_forest.pkl
[iterative] gradient_boosting 완료: 저장 -> iterative_gradient_boosting.pkl
[iterative] xgboost 완료: 저장 -> iterative_xgboost.pkl


,dataset,model,train_accuracy,train_precision,train_recall,train_f1,train_roc_auc,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
8,iterative,gradient_boosting,0.960870,0.993056,0.89375,0.940789,0.993542,0.779221,0.700000,0.648148,0.673077,0.867130,0.714286,0.613636,0.500000,0.551020,0.799259
5,iterative,log_reg,0.778261,0.666667,0.72500,0.694611,0.850479,0.759740,0.649123,0.685185,0.666667,0.850926,0.733766,0.606557,0.685185,0.643478,0.812407
7,iterative,random_forest,1.000000,1.000000,1.00000,1.000000,1.000000,0.779221,0.708333,0.629630,0.666667,0.869907,0.727273,0.630435,0.537037,0.580000,0.810741
9,iterative,xgboost,0.991304,0.993671,0.98125,0.987421,0.999708,0.759740,0.688889,0.574074,0.626263,0.837593,0.746753,0.653061,0.592593,0.621359,0.811667
6,iterative,knn,0.810870,0.748299,0.68750,0.716612,0.898479,0.753247,0.673913,0.574074,0.620000,0.809630,0.746753,0.641509,0.629630,0.635514,0.777963
1,median,knn,0.817391,0.771429,0.67500,0.720000,0.901240,0.785714,0.714286,0.648148,0.679612,0.842222,0.740260,0.640000,0.592593,0.615385,0.769259
2,median,random_forest,1.000000,1.000000,1.00000,1.000000,1.000000,0.785714,0.714286,0.648148,0.679612,0.868241,0.740260,0.659091,0.537037,0.591837,0.824537
0,median,log_reg,0.782609,0.674419,0.72500,0.698795,0.847896,0.753247,0.637931,0.685185,0.660714,0.852407,0.727273,0.596774,0.685185,0.637931,0.814815
3,median,gradient_boosting,0.958696,0.973154,0.90625,0.938511,0.991938,0.766234,0.680000,0.629630,0.653846,0.843148,0.727273,0.630435,0.537037,0.580000,0.805926
4,median,xgboost,0.984783,0.993548,0.96250,0.977778,0.999521,0.746753,0.641509,0.629630,0.635514,0.841667,0.753247,0.673913,0.574074,0.620000,0.819259
